# Asymmetric Qubitization

This notebook is an example of how to use the asymmetric qubitization functionality, as well as use the resulting quantum walk with QSP.

In [ ]:
from psiqworkbench import Qubits, QPU
from psiqworkbench.integrations import circuit_designer
import numpy as np
from syk_simulation.qubitization.qsp import qsp_evolution
from syk_simulation.qubitization.asymmetric_qubitization import AsymmetricQubitization

N = 4 
random_seed = 3
time = 1
epsilon = 1e-3
J=1

# Qubit register sizes
system_size = N
index_chunk = int(np.ceil(np.log2(N)))
index_size = 4 * index_chunk
aux_unary_size = index_chunk
range_flag_size = 1
branch_size = 1
mode_size = 1
selection_size = 1

walk_size = branch_size + index_size + system_size
clean_ladder_aux = 10
num_qubits = walk_size + aux_unary_size + range_flag_size + mode_size + selection_size + clean_ladder_aux


# Setup QPU and Qubit registers
# (except for aux_unary and range_flag as they are auxiliary Qubrick qubits)

# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>",">>clean-ladder-filter>>",">>single-control-filter>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>",">>clean-ladder-filter>>",">>toffoli-filter>>",">>single-control-filter>>"])

# qpu = QPU(num_qubits=num_qubits, filters=NO_SIM_LONG)

qpu= QPU(num_qubits=num_qubits, filters=[">>buffer>>"])

# qpu = QPU(num_qubits=100, filters=[">>clean-ladder-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>toffoli-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>single-control-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>", ">>single-control-filter>>", ">>buffer>>"])

# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>", ">>buffer>>"]) #">>single-control-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>single-control-filter>>"])

# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>",">>single-control-filter>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>",">>single-control-filter>>", ">>rs-synth-filter>>"])

walk = Qubits(walk_size, "walk", qpu)
branch = Qubits(walk[0:branch_size], "branch")
index = Qubits(walk[branch_size : index_size + branch_size], "index")
system = Qubits(walk[index_size + branch_size :], "system")
mode = Qubits(mode_size, "mode", qpu=qpu)
selection = Qubits(selection_size, "selection", qpu=qpu)


# PREPROCESSING
# 1) set branch |+>
branch.had()
qsp_evolution(N=N, J=J, branch=branch, index=index, system=system, mode=mode, selection=selection, time=time, epsilon=epsilon, random_seed=random_seed)


branch.had()

circuit_designer.draw(qpu)